# 02 — Data Cleaning

Clean `Data/Raw/Data_input.xlsx` và xuất `Data/Processed/Data_processed.xlsx`.

Nguyên tắc:
- Giữ nguyên 14 sheet cần cho workflow mới.
- Strip text + bỏ exact duplicate rows.
- Coerce numeric fields có schema rõ ràng.
- **Không fill missing bằng 0.**
- Không biến status/guardrail thành numeric score.
- Không chọn Top 3.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

input_path = Path("../Data/Raw/Data_input.xlsx")
output_dir = Path("../Data/Processed")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "Data_processed.xlsx"

# Canonical names used by the project.
sheet_prefixes = {
    "01_Current_Admin": "01_Current_Admin",
    "02_Evidence_Log": "02_Evidence_Log",
    "03_District_Profile": "03_District_Profile",
    "04_Market_Locations": "04_Market_Locations",
    "05_District_Market_Footprint": "05_District_Market_Footprint",
    "06_iCare_Coverage_Evidence": "06_iCare_Coverage_Evidence",
    "07_District_Traffic_Evidence": "07_District_Traffic_Evidence",
    "08_District_Traffic_Summary": "08_District_Traffic_Summary",
    "09_District_Opportunity_Base": "09_District_Opportunity_Base",
    "10_Gaps_Next": "10_Gaps_Next",
    "11_Old_Project_Universe_Research": "11_Old_Project_Universe_Resear",
    "12_Old_Project_Universe_Method": "12_Old_Project_Universe_Metho",
    "13_Additional_Context_Evidence": "13_Additional_Context_Evidenc",
    "14_Complete_Screening_Core": "14_Complete_Screening_Core",
}

assert input_path.exists(), f"Input not found: {input_path.resolve()}"

xls = pd.ExcelFile(input_path)

def resolve_sheet_name(canonical_name, prefix):
    if canonical_name in xls.sheet_names:
        return canonical_name
    matches = [s for s in xls.sheet_names if s.startswith(prefix)]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError(f"Ambiguous sheet prefix {prefix!r}: {matches}")
    raise KeyError(f"Sheet not found for {canonical_name!r}; expected prefix {prefix!r}")

resolved_sheets = {
    canonical: resolve_sheet_name(canonical, prefix)
    for canonical, prefix in sheet_prefixes.items()
}

sheet_names = list(sheet_prefixes.keys())

raw = {
    canonical: pd.read_excel(input_path, sheet_name=actual)
    for canonical, actual in resolved_sheets.items()
}

print("Loaded:", input_path.resolve())
print("\nResolved Excel sheet names:")
for canonical, actual in resolved_sheets.items():
    if canonical != actual:
        print(f"- {canonical} -> {actual}")


Loaded: D:\IcareWork\Place for icare\Data\Raw\Data_input.xlsx

Resolved Excel sheet names:
- 11_Old_Project_Universe_Research -> 11_Old_Project_Universe_Researc


In [2]:
def clean_strings(df):
    out = df.copy()

    for col in out.columns:
        if out[col].dtype == "object":
            out[col] = out[col].map(
                lambda x: x.strip() if isinstance(x, str) else x
            )
            out[col] = out[col].replace({"": pd.NA})

    # Keep first occurrence of exact duplicate row.
    out = out.drop_duplicates().reset_index(drop=True)
    return out

clean = {name: clean_strings(df) for name, df in raw.items()}

print("String cleaning complete.")


String cleaning complete.


## Numeric coercion

Chỉ coerce các field định nghĩa là numeric. Invalid text → `NaN`, không → 0.


In [3]:
numeric_columns = {
    "01_Current_Admin": [
        "Diện tích (km²)",
        "Dân số dùng để sắp xếp",
        "Mật độ (người/km²)",
    ],
    "04_Market_Locations": [
        "latitude","longitude",
    ],
    "05_District_Market_Footprint": [
        "total_verified_locations",
        "verified_competitor_locations_excl_icare",
        "verified_competitor_brands_excl_icare",
        "icare_verified_locations",
        "location_share_pct_all_brands",
        "competition_rank_old_project",
        "top_brand_location_count",
    ],
    "06_iCare_Coverage_Evidence": [
        "latitude","longitude","nearest_icare_km",
    ],
    "08_District_Traffic_Summary": [
        "Official/credible evidence rows",
        "Distinct source URLs",
        "Distinct corridors/nodes",
        "Congestion/flow-management evidence rows",
        "Gateway/hub/heavy-vehicle evidence rows",
    ],
    "09_District_Opportunity_Base": [
        "competitor_locations",
        "competitor_brands",
        "icare_locations",
        "median_nearest_icare_km",
        "uncovered_share_5km_pct",
        "population_reference_year",
        "official_population",
        "official_density",
        "official_acting_enterprises_2022",
        "official_enterprise_turnover_2022_billion_vnd",
        "official_avg_employee_compensation_2022_million_month",
        "traffic_evidence_rows",
        "traffic_distinct_sources",
        "traffic_distinct_corridors_nodes",
        "apartment_midpoint_proxy",
    ],
    "11_Old_Project_Universe_Research": [
        "verified_competitor_locations",
        "verified_competitor_brands",
        "icare_verified_locations",
        "median_nearest_icare_km_at_market_points",
        "observed_market_points_beyond_5km_pct",
        "population_reference_year",
        "official_area_km2",
        "official_population",
        "official_density",
        "official_acting_enterprises_2022",
        "official_enterprise_turnover_2022_billion_vnd",
        "official_avg_employee_compensation_2022_million_month",
        "traffic_evidence_rows",
        "traffic_distinct_sources",
        "traffic_distinct_corridors_nodes",
        "apartment_midpoint_proxy",
    ],
    "13_Additional_Context_Evidence": [
        "year",
    ],
    "14_Complete_Screening_Core": [
        "competitor_locations",
        "competitor_brands",
        "icare_locations",
        "population_value",
        "population_reference_year",
        "traffic_evidence_rows",
    ],
}

for sheet_name, cols in numeric_columns.items():
    df = clean[sheet_name]
    for col in cols:
        if col not in df.columns:
            print(f"WARNING: {sheet_name}: numeric column missing -> {col}")
            continue
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Numeric coercion complete.")


Numeric coercion complete.


## Preserve flags / guardrails

Không convert các status thành score. Chỉ kiểm tra format hợp lý.


In [4]:
opp = clean["09_District_Opportunity_Base"]

status_cols = [
    "market_data_available",
    "coverage_data_available",
    "traffic_data_available",
    "economic_data_available",
    "pp_data_available",
    "screening_readiness",
    "pp_comparability_flag",
    "population_source_quality",
]

print("=== OPPORTUNITY FLAGS ===")
for col in status_cols:
    if col in opp.columns:
        print(f"\n{col}:")
        print(opp[col].value_counts(dropna=False))

core = clean["14_Complete_Screening_Core"]

print("\n=== CORE STATUS FIELDS ===")
for col in [
    "coverage_status",
    "population_quality",
    "traffic_evidence_status",
    "economic_comparability_status",
    "income_reference_status",
    "pp_proxy_status",
    "screening_core_complete",
]:
    print(f"\n{col}:")
    print(core[col].value_counts(dropna=False))


=== OPPORTUNITY FLAGS ===

market_data_available:
market_data_available
YES    19
Name: count, dtype: int64

coverage_data_available:
coverage_data_available
YES                     18
DIRECT_ICARE_PRESENT     1
Name: count, dtype: int64

traffic_data_available:
traffic_data_available
YES    19
Name: count, dtype: int64

economic_data_available:
economic_data_available
YES                               15
YES_WITH_SOURCE_HOST_GUARDRAIL     2
PARTIAL                            2
Name: count, dtype: int64

pp_data_available:
pp_data_available
YES    18
NO      1
Name: count, dtype: int64

screening_readiness:
screening_readiness
READY_FOR_COMPARATIVE_SCREENING                      15
READY_WITH_CROSS_PROVINCE_COMPARABILITY_GUARDRAIL     2
PARTIAL                                               2
Name: count, dtype: int64

pp_comparability_flag:
pp_comparability_flag
COMPARABLE_WITH_CAUTION                  12
LOW_COMPARABILITY_LUXURY_OUTLIER_RISK     3
LOW_COMPARABILITY_MIXED_SUBMARKETS   

## QA after cleaning


In [5]:
print("=== CLEANING QA ===")

for name, df in clean.items():
    print(
        f"{name:35s}",
        "rows=", len(df),
        "duplicates=", int(df.duplicated().sum())
    )

expected_universe = {
    "Thủ Đức", "Tân Bình", "Bình Thạnh", "Quận 10", "Quận 1",
    "Quận 12", "Quận 7", "Bình Tân", "Gò Vấp", "Quận 3",
    "Tân Phú", "Dĩ An", "Quận 4", "Quận 6", "Thủ Dầu Một",
    "Vũng Tàu", "Bà Rịa", "Phú Nhuận", "Quận 5"
}

assert set(clean["09_District_Opportunity_Base"]["district_old"].dropna()) == expected_universe
assert set(clean["14_Complete_Screening_Core"]["district_old"].dropna()) == expected_universe

locations = clean["04_Market_Locations"]
coverage = clean["06_iCare_Coverage_Evidence"]

competitor_count = int(locations["analysis_role"].eq("COMPETITOR_EVIDENCE").sum())

assert len(coverage) == competitor_count, (
    f"Coverage rows ({len(coverage)}) != competitor locations ({competitor_count})"
)
assert coverage["nearest_icare_km"].notna().all()
assert not coverage["nearest_icare_km"].lt(0).any()

print("\nUniverse preserved:", len(expected_universe), "areas")
print("Coverage rows:", len(coverage))
print("Core rows:", len(core))
print("Opportunity rows:", len(opp))


=== CLEANING QA ===
01_Current_Admin                    rows= 20 duplicates= 0
02_Evidence_Log                     rows= 4 duplicates= 0
03_District_Profile                 rows= 5 duplicates= 0
04_Market_Locations                 rows= 65 duplicates= 0
05_District_Market_Footprint        rows= 19 duplicates= 0
06_iCare_Coverage_Evidence          rows= 62 duplicates= 0
07_District_Traffic_Evidence        rows= 36 duplicates= 0
08_District_Traffic_Summary         rows= 19 duplicates= 0
09_District_Opportunity_Base        rows= 19 duplicates= 0
10_Gaps_Next                        rows= 6 duplicates= 0
11_Old_Project_Universe_Research    rows= 19 duplicates= 0
12_Old_Project_Universe_Method      rows= 8 duplicates= 0
13_Additional_Context_Evidence      rows= 23 duplicates= 0
14_Complete_Screening_Core          rows= 19 duplicates= 0

Universe preserved: 19 areas
Coverage rows: 62
Core rows: 19
Opportunity rows: 19


## Missing-value QA

Missing numeric vẫn được phép khi source không comparable.  
Không được auto-fill thành 0.


In [6]:
check_numeric = [
    "median_nearest_icare_km",
    "uncovered_share_5km_pct",
    "official_population",
    "official_density",
    "official_acting_enterprises_2022",
    "official_enterprise_turnover_2022_billion_vnd",
    "official_avg_employee_compensation_2022_million_month",
    "apartment_midpoint_proxy",
]

missing_report = pd.DataFrame({
    "column": check_numeric,
    "missing_count": [int(opp[c].isna().sum()) for c in check_numeric],
})

display(missing_report)

print("\nNo missing value has been filled with zero by this notebook.")


,column,missing_count
0,median_nearest_icare_km,1
1,uncovered_share_5km_pct,1
2,official_population,0
3,official_density,0
4,official_acting_enterprises_2022,2
5,official_enterprise_turnover_2022_billion_vnd,2
6,official_avg_employee_compensation_2022_millio...,2
7,apartment_midpoint_proxy,1



No missing value has been filled with zero by this notebook.


## Export `Data_processed.xlsx`

Giữ **cùng sheet names** để `03_EDA.ipynb` đọc trực tiếp.


In [7]:
# Excel sheet names are capped at 31 characters.
output_sheet_names = {
    "01_Current_Admin": "01_Current_Admin",
    "02_Evidence_Log": "02_Evidence_Log",
    "03_District_Profile": "03_District_Profile",
    "04_Market_Locations": "04_Market_Locations",
    "05_District_Market_Footprint": "05_District_Market_Footprint",
    "06_iCare_Coverage_Evidence": "06_iCare_Coverage_Evidence",
    "07_District_Traffic_Evidence": "07_District_Traffic_Evidence",
    "08_District_Traffic_Summary": "08_District_Traffic_Summary",
    "09_District_Opportunity_Base": "09_District_Opportunity_Base",
    "10_Gaps_Next": "10_Gaps_Next",
    "11_Old_Project_Universe_Research": "11_Old_Project_Research",
    "12_Old_Project_Universe_Method": "12_Old_Project_Method",
    "13_Additional_Context_Evidence": "13_Context_Evidence",
    "14_Complete_Screening_Core": "14_Screening_Core",
}

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for canonical in sheet_names:
        clean[canonical].to_excel(
            writer,
            sheet_name=output_sheet_names[canonical],
            index=False
        )

print("Saved:", output_path.resolve())


Saved: D:\IcareWork\Place for icare\Data\Processed\Data_processed.xlsx


## Post-export verification


In [8]:
check_xls = pd.ExcelFile(output_path)

expected_output_sheets = list(output_sheet_names.values())
missing_after_export = [s for s in expected_output_sheets if s not in check_xls.sheet_names]
assert not missing_after_export, f"Missing after export: {missing_after_export}"

check_opp = pd.read_excel(output_path, sheet_name=output_sheet_names["09_District_Opportunity_Base"])
check_core = pd.read_excel(output_path, sheet_name=output_sheet_names["14_Complete_Screening_Core"])

assert len(check_opp) == 19
assert len(check_core) == 19
assert check_core["district_old"].nunique() == 19

print("Post-export verification OK.")
print("Next: run 03_EDA.ipynb")


Post-export verification OK.
Next: run 03_EDA.ipynb
